In [7]:
!pip install -qU llama-index llama-index-vector-stores-chroma llama-index-embeddings-huggingface llama-index-llms-openai llama-index-postprocessor-flag-embedding-reranker chromadb gradio

In [8]:
import os

os.makedirs("enterprise_data", exist_ok=True)

documents = {
    "enterprise_data/policy.txt": """Company Security & Data Privacy Policy

1. Access Control: All employees must use multi-factor authentication (MFA) for accessing core internal databases.
2. Remote Work Protocol: Mandatory encrypted VPN connections must be active whenever accessing corporate assets outside office network boundaries.
3. Data Retention: Customer audit logs are retained for 365 days in cold storage before being permanently purged.""",

    "enterprise_data/architecture.txt": """System Architecture & Production API Standards

1. Microservices Architecture: Internal web services communicate asynchronously using gRPC over HTTP/2.
2. Database Strategy: Primary transactional workloads use PostgreSQL, while semantic vector retrieval runs on persistent ChromaDB clusters.
3. SLA Metrics: API target latency must maintain less than 200 milliseconds for p95 request traffic.""",

    "enterprise_data/compliance.txt": """Global Regulatory Compliance Framework

1. GDPR Compliance: User account deletion requests (Right to be Forgotten) must be fully executed within 30 days of submission.
2. SOC 2 Type II: Automated logging pipelines must record all administrative infrastructure interactions.
3. Encryption Standards: All data at rest must be encrypted using AES-256 standard, and data in transit must enforce TLS 1.3."""
}

for path, content in documents.items():
    with open(path, "w") as f:
        f.write(content)

In [ ]:
import os
import getpass
from llama_index.core import Settings, SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OpenAI API Key: ")

Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0.0)
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.node_parser = SentenceSplitter(chunk_size=256, chunk_overlap=32)

reader = SimpleDirectoryReader("enterprise_data")
raw_docs = reader.load_data()
nodes = Settings.node_parser.get_nodes_from_documents(raw_docs)

In [ ]:
import chromadb
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.vector_stores.chroma import ChromaVectorStore

db_client = chromadb.PersistentClient(path="./enterprise_chroma")
collection = db_client.get_or_create_collection("production_knowledge_base")

vector_store = ChromaVectorStore(chroma_collection=collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex(nodes, storage_context=storage_context)

In [ ]:
from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.core.prompts import PromptTemplate

reranker = SentenceTransformerRerank(
    model="BAAI/bge-reranker-base",
    top_n=2
)

qa_template_str = (
    "Context information is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Given the context information and strictly no prior knowledge, "
    "answer the user query in a concise, authoritative manner. "
    "If the context does not contain the answer, respond: "
    "'The requested information is not present in the enterprise knowledge base.'\n\n"
    "Query: {query_str}\n"
    "Answer: "
)

qa_template = PromptTemplate(qa_template_str)

query_engine = index.as_query_engine(
    similarity_top_k=5,
    node_postprocessors=[reranker],
    text_qa_template=qa_template
)

In [ ]:
from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.core.prompts import PromptTemplate

reranker = SentenceTransformerRerank(
    model="BAAI/bge-reranker-base",
    top_n=2
)

qa_template_str = (
    "Context information is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Given the context information and strictly no prior knowledge, "
    "answer the user query in a concise, authoritative manner. "
    "If the context does not contain the answer, respond: "
    "'The requested information is not present in the enterprise knowledge base.'\n\n"
    "Query: {query_str}\n"
    "Answer: "
)

qa_template = PromptTemplate(qa_template_str)

query_engine = index.as_query_engine(
    similarity_top_k=5,
    node_postprocessors=[reranker],
    text_qa_template=qa_template
)

In [ ]:
import gradio as gr

def process_query(message, history):
    response = query_engine.query(message)
    answer = str(response)

    citations = "\n\n---\n### Verified Source References:\n"
    for i, node in enumerate(response.source_nodes, 1):
        file_name = node.metadata.get("file_name", "Unknown File")
        score = round(node.score, 4) if node.score is not None else "N/A"
        snippet = node.node.get_content().replace("\n", " ")
        citations += f"**[{i}] {file_name}** (Reranker Score: {score})\n> \"{snippet}\"\n\n"

    return answer + citations

demo = gr.ChatInterface(
    fn=process_query,
    title="Production Enterprise RAG Platform",
    description="Engineered with LlamaIndex, persistent ChromaDB vector storage, BGE Embeddings, BGE Cross-Encoder Reranking, and deterministic grounding constraints.",
    theme="soft"
)

demo.launch(share=True)